# Lab 1: Text Representation and Language Models

## At a glance

Shrink a word vocabulary and a language model's perplexity goes down. Someone reading only that number would call
the smaller vocabulary the better choice. In this lab you test that judgment on twelve Sherlock Holmes stories: you
train the same bigram model on text cut into tokens in different ways, and then decide which comparison is fair.

You write four short functions, choose two tokenizers of your own and predict how they will compare, and then run a
second comparison that checkpoint 3 asks about. Everything runs on a CPU; the slowest cell takes under a minute.

> Save your own copy first: File → Save a copy in Drive.

> ### <font color="#A31F34">Big picture</font>
>
> Perplexity per token measures how surprised a model is, on average, by each token. What counts as a token is a
> choice made before any training starts. This lab asks what that choice does to the number, and what a number would
> need in order to compare two tokenizers at all.

## What is graded

- Checkpoint 1, readiness (20%): C1 to C3 from the Changed cases cell, and D1 and D2 from the Report values cell.
- Checkpoint 2, experiment record (30%): what a record of your runs has to hold for someone else to check the
  comparison.
- Checkpoint 3, claim, evidence and caveat (35%): which statements a second comparison supports, run with two merge
  counts that checkpoint 3 gives you.
- Checkpoint 4, next experiment (15%): what has to stay fixed when the comparison is repeated on other stories.

The numbers from your own two tokenizers depend on what you choose, so they are not auto-graded. They are there to
test your predictions.

> ### <font color="#B45309">Watch out</font>
>
> λ is chosen on the validation story. Look at the test story only through the two scored tables, once your tokenizers
> are fixed: a setting picked by looking at the test story makes its test score an overestimate.

In [ ]:
import base64
import gzip
import hashlib
import math
import re
import time
from collections import Counter

In [ ]:
#@title Load the stories (run this cell; there is no need to read it)
BLOB = "..."
TEXT = gzip.decompress(base64.b64decode(BLOB)).decode("utf-8")
assert hashlib.sha256(TEXT.encode("utf-8")).hexdigest() == "e4dbbd992e8aa33750ad826e353d506884d3edf4caa1a9edc2ecfec9889d5d8f", "The stories did not load intact. Run this cell again."
STORIES = TEXT.split("\n\n")
print(len(STORIES), "stories loaded")

## Setup: the stories and the split

The text is Arthur Conan Doyle's *The Adventures of Sherlock Holmes* (1892), lowercased, with punctuation simplified.
The first ten stories train every model. The eleventh is the validation story, used only to choose λ. The twelfth is
the test story, which none of the models sees until it is scored. Each split is read as one long sequence, as in
lecture 1.2.

Word-level tokenizers split the text into words and punctuation marks.

In [ ]:
TRAIN, VALID, TEST = " ".join(STORIES[:10]), STORIES[10], STORIES[11]
ALPHABET = sorted(set(TRAIN))
WORD = re.compile(r"[a-z0-9]+|[^\sa-z0-9]")

TRAIN_WORDS, VALID_WORDS, TEST_WORDS = WORD.findall(TRAIN), WORD.findall(VALID), WORD.findall(TEST)
print(f"training: {len(TRAIN_WORDS):,} words, {len(set(TRAIN_WORDS)):,} different ones; "
      f"test: {len(TEST_WORDS):,} words, {len(TEST):,} characters; alphabet of {len(ALPHABET)} characters")
print("the test story begins:", " ".join(TEST_WORDS[:18]))

## Your task 1: Map words to a fixed vocabulary

A word-level tokenizer keeps the K most frequent training words, and every other word becomes the single token
`<unk>`. Write `to_vocabulary`, which does the mapping, and `unknown_rate`, the fraction of words that fall outside
the vocabulary. Count repeats: a word missing three times counts three times.

In [ ]:
UNK = "<unk>"

def build_vocabulary(words, k):
    """The k most frequent words; ties go to the word seen first."""
    return {w for w, _ in Counter(words).most_common(k)}

In [ ]:
def to_vocabulary(words, vocab):
    """Return `words` with every word outside `vocab` replaced by UNK."""
    # TODO: keep each word that is in the vocabulary and replace the rest with UNK.
    raise NotImplementedError("Map the words.")

def unknown_rate(words, vocab):
    """Return the fraction of `words`, counting repeats, that are not in `vocab`."""
    # TODO: count the words outside the vocabulary, repeats included, and divide by how many words there are.
    raise NotImplementedError("Compute the unknown rate.")

In [ ]:
toy = "the dog saw the other dog".split()
assert to_vocabulary(toy, {"the", "dog"}) == ["the", "dog", UNK, "the", UNK, "dog"]
assert unknown_rate(toy, {"the", "dog"}) == 2 / 6
print("Task 1 checks pass.")

## Your task 2: Score a token sequence in bits

A bigram model with add-λ smoothing, as in lecture 1.2, gives each token a probability given the token before it.
`sequence_bits` returns the total cost of a sequence in bits: minus the base-2 log of every factor, added up, starting
from the second token because the first is given.

<details><summary>Hint</summary>

The factor for `nxt` after `prev` is (count of the pair + λ) / (count of `prev` + λ × V), where V is the vocabulary
size. `math.log2` gives base-2 logs.

</details>

In [ ]:
def count_bigrams(tokens):
    """Counts of each neighboring pair, and of each token that has a token after it."""
    return Counter(zip(tokens, tokens[1:])), Counter(tokens[:-1])

In [ ]:
def sequence_bits(tokens, pairs, contexts, vocab_size, lam):
    """Total cost of `tokens` in bits under add-lambda bigram estimates, the first token given."""
    # TODO: add up minus log2 of each factor, for every pair of neighboring tokens.
    raise NotImplementedError("Score the sequence.")

In [ ]:
pairs, contexts = count_bigrams("a b a b".split())
assert abs(sequence_bits(["a", "b"], pairs, contexts, 2, 0) - 0.0) < 1e-12            # b always followed a
assert abs(sequence_bits(["b", "b"], pairs, contexts, 2, 1) - math.log2(3)) < 1e-12    # (0 + 1) / (1 + 2)
many = "x y z x y".split()
assert abs(sequence_bits(many, *count_bigrams(many), 3, 1e9) - 4 * math.log2(3)) < 1e-6, "A huge lambda should cost log2(V) bits per scored token."
print("Task 2 checks pass.")

## Your task 3: Report the score three ways

`summarize` turns a sequence's cost into three numbers:

- **perplexity per token**: 2 raised to the average cost of a scored token, where every token but the first is scored;
- **bits per character**: the total cost divided by the number of characters in the text;
- **bits per character, unknown words charged**: before dividing, add the cost of spelling out every unknown word one
  character at a time, with each of the alphabet's characters equally likely, which is length × log2(alphabet size)
  bits for each unknown word.

Each number answers a different question. Perplexity per token divides by the number of tokens, and how many tokens
a text has depends on the tokenizer. Bits per character divides by the length of the text, which no tokenizer
changes. The charge puts back a cost that a word model skips: when a word becomes `<unk>`, the model pays for
`<unk>`, not for that word, and spelling the word out is what it would take to say which word it was.

<details><summary>Worked example</summary>

A word model spends 40 bits on a text of 9 tokens and 50 characters. One of its words, "zoo", is unknown, and the
alphabet has 8 characters. Perplexity per token is 2^(40/8) = 32, since 8 of the 9 tokens are scored. Bits per
character is 40/50 = 0.8. The charge adds 3 × log2 8 = 9 bits, so bits per character with unknown words charged is
49/50 = 0.98.

</details>

In [ ]:
def summarize(bits, n_tokens, n_chars, unknown_words, alphabet_size):
    """Return a dict with per_token_perplexity, bits_per_char and bits_per_char_charged."""
    # TODO: compute the three numbers described above.
    raise NotImplementedError("Summarize the score.")

In [ ]:
s = summarize(bits=30.0, n_tokens=11, n_chars=60, unknown_words=["ab"], alphabet_size=4)
assert abs(s["per_token_perplexity"] - 2 ** 3) < 1e-12
assert abs(s["bits_per_char"] - 0.5) < 1e-12
assert abs(s["bits_per_char_charged"] - (30 + 2 * 2) / 60) < 1e-12
print("Task 3 checks pass.")

## Changed cases

Before trusting your functions on the stories, run them on inputs they have not seen. Replace the values at the top of
the next cell with the ones on the course page for checkpoint 1, run it, and enter C1 to C3 as it prints them.

In [ ]:
# Replace these values with the inputs from the course page, then run the cell.
C1_VOCABULARY = {"the", "man", "."}
C1_WORDS = "the man ran .".split()
C2_TRAINING_TOKENS = "a b a".split()
C2_SEQUENCE = "a b".split()
C2_VOCABULARY_SIZE = 2
C2_LAMBDA = 1.0
C3_BITS, C3_TOKENS, C3_CHARACTERS = 10.0, 5, 20
C3_UNKNOWN_WORDS = ["cab"]
C3_ALPHABET_SIZE = 8

print("C1 =", round(unknown_rate(C1_WORDS, C1_VOCABULARY), 6))
print("C2 =", round(sequence_bits(C2_SEQUENCE, *count_bigrams(C2_TRAINING_TOKENS), C2_VOCABULARY_SIZE, C2_LAMBDA), 6))
print("C3 =", round(summarize(C3_BITS, C3_TOKENS, C3_CHARACTERS, C3_UNKNOWN_WORDS, C3_ALPHABET_SIZE)["bits_per_char_charged"], 6))

## Provided: BPE at the scale of a book

This is the byte pair encoding from your coding practice, with one change for speed: instead of counting pairs across
the whole text for every merge, it counts each distinct word once and weights it by how often the word occurs. That
gives exactly the same merges, in the same order, because every occurrence of a word is split the same way. Encoding
still applies the merges in the order they were learned.

In [ ]:
PIECE = re.compile(r" ?[a-z0-9]+| ?[^\sa-z0-9]+|\s+")

def _merge(seq, a, b):
    out, i = [], 0
    while i < len(seq):
        if i + 1 < len(seq) and seq[i] == a and seq[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(seq[i]); i += 1
    return out

def train_bpe(text, num_merges):
    words = Counter(PIECE.findall(text))
    split = {w: list(w) for w in words}
    merges = []
    for _ in range(num_merges):
        pairs = Counter()
        for w, n in words.items():
            s = split[w]
            for pair in zip(s, s[1:]):
                pairs[pair] += n
        if not pairs:
            break
        (a, b), n = max(pairs.items(), key=lambda item: item[1])
        if n < 2:
            break
        merges.append((a, b))
        for w in words:
            if a in split[w]:
                split[w] = _merge(split[w], a, b)
    return merges

def encode(text, merges):
    cache, out = {}, []
    for piece in PIECE.findall(text):
        if piece not in cache:
            s = list(piece)
            for a, b in merges:
                if len(s) < 2:
                    break
                if a in s:
                    s = _merge(s, a, b)
            cache[piece] = s
        out.extend(cache[piece])
    return out

## Provided: the runs and the shared control

Every run trains the same bigram model and chooses λ from the same grid on the validation story, so the only thing
that changes from run to run is the tokenizer. The control is a word-level tokenizer that keeps every word seen at
least twice in the training stories. Words seen only once become `<unk>` during training, and that is how the model
learns how often a new word turns up. A vocabulary of every training word would never see `<unk>` in training, and
would give each unknown word in the test story almost no probability.

In [ ]:
LAMBDAS = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

def run(label, train_tokens, valid_tokens, test_tokens, vocab_size, unknown_test_words):
    pairs, contexts = count_bigrams(train_tokens)
    lam = min(LAMBDAS, key=lambda l: sequence_bits(valid_tokens, pairs, contexts, vocab_size, l))
    bits = sequence_bits(test_tokens, pairs, contexts, vocab_size, lam)
    row = summarize(bits, len(test_tokens), len(TEST), unknown_test_words, len(ALPHABET))
    row.update(label=label, vocab_size=vocab_size, lam=lam, test_tokens=len(test_tokens),
               unknown_rate=len(unknown_test_words) / len(TEST_WORDS))
    return row

def timed(make, *args):
    start = time.perf_counter()
    row = make(*args)
    row["seconds"] = time.perf_counter() - start
    return row

def run_words(k):
    vocab = build_vocabulary(TRAIN_WORDS, k)
    return run(f"words, top {k}", to_vocabulary(TRAIN_WORDS, vocab), to_vocabulary(VALID_WORDS, vocab),
               to_vocabulary(TEST_WORDS, vocab), len(vocab) + 1, [w for w in TEST_WORDS if w not in vocab])

def run_bpe(merges):
    return run(f"BPE, {len(merges)} merges", encode(TRAIN, merges), encode(VALID, merges), encode(TEST, merges),
               len(ALPHABET) + len(merges), [])

CONTROL_SIZE = sum(n >= 2 for n in Counter(TRAIN_WORDS).values())
control = timed(run_words, CONTROL_SIZE)
control["label"] += " (control)"
print(f"control: {control['label']}, unknown rate {control['unknown_rate']:.3f}, "
      f"perplexity per token {control['per_token_perplexity']:.1f}")

def show(rows):
    print(f"{'run':26s} {'vocab':>6s} {'unknown':>8s} {'lambda':>7s} {'test tokens':>11s} {'perplexity':>11s} "
          f"{'bits/char':>10s} {'charged':>8s} {'seconds':>8s}")
    for r in rows:
        print(f"{r['label']:26s} {r['vocab_size']:6d} {r['unknown_rate']:8.3f} {r['lam']:7g} {r['test_tokens']:11,d} "
              f"{r['per_token_perplexity']:11.1f} {r['bits_per_char']:10.3f} {r['bits_per_char_charged']:8.3f} "
              f"{r['seconds']:8.2f}")

## Your task 4: Choose two tokenizers

Choose a word vocabulary smaller than the control's, from 100 to 3,500 words, and a number of BPE merges, from 100 to
4,000. Each one changes a single thing from the control: how the text is cut into tokens.

Predict before you run the record: which of your two tokenizers will have the lower perplexity per token, and which
the lower bits per character with unknown words charged? Enter each guess as `"words"` or `"bpe"`. The record prints
your guesses beside what happened.

In [ ]:
# STUDENT TASK 4: replace None with your two choices and your two predictions.
MY_WORD_VOCABULARY = None   # from 100 to 3,500
MY_MERGES = None            # from 100 to 4,000
LOWER_PERPLEXITY = None     # "words" or "bpe"
LOWER_CHARGED = None        # "words" or "bpe"

In [ ]:
assert isinstance(MY_WORD_VOCABULARY, int) and 100 <= MY_WORD_VOCABULARY <= 3500, "Choose a word vocabulary from 100 to 3,500."
assert isinstance(MY_MERGES, int) and 100 <= MY_MERGES <= 4000, "Choose from 100 to 4,000 merges."
assert LOWER_PERPLEXITY in ("words", "bpe") and LOWER_CHARGED in ("words", "bpe"), 'Enter each prediction as "words" or "bpe".'
MERGES = train_bpe(TRAIN, max(MY_MERGES, 1000))      # the first k of these merges are exactly a k-merge tokenizer
mine = {"words": timed(run_words, MY_WORD_VOCABULARY), "bpe": timed(run_bpe, MERGES[:MY_MERGES])}

print("Evidence Record: the same bigram model with add-lambda smoothing on every run")
print(f"trained on stories 1 to 10; lambda chosen on story 11 from {LAMBDAS}; scored on story 12")
print("perplexity is per test token; bits/char is per character of the test story; charged adds the unknown words' spelling")
print()
show([control, mine["words"], mine["bpe"]])
print()
for guess, key, what in [(LOWER_PERPLEXITY, "per_token_perplexity", "perplexity per token"),
                         (LOWER_CHARGED, "bits_per_char_charged", "charged bits per character")]:
    lower = min(mine, key=lambda name: mine[name][key])
    print(f"lower {what}: you predicted {guess}, the record shows {lower}")

## A second comparison, for checkpoint 3

Checkpoint 3 gives you two numbers of BPE merges. Replace the values at the top of the next cell with them and run it:
it scores both tokenizers against the same control.

Predict first: will either tokenizer beat the control on bits per character with unknown words charged?

In [ ]:
# Replace these with the two merge counts from the course page for checkpoint 3, then run the cell.
CHECK_MERGES = [500, 2000]

assert all(isinstance(m, int) and 100 <= m <= 4000 for m in CHECK_MERGES), "Use merge counts from 100 to 4,000."
check_merges = train_bpe(TRAIN, max(CHECK_MERGES))
show([control] + [timed(run_bpe, check_merges[:m]) for m in CHECK_MERGES])

## Read the evidence

Before you open checkpoints 3 and 4, read both tables three ways.

1. The test-token column. Which runs cut the test story into the same number of tokens, and what differs between
   them? When two runs have different numbers of tokens, what is a comparison of their perplexities per token
   comparing?
2. The spaces. Run `print(encode(TEST[:60], MERGES))` and `print(TEST_WORDS[:10])` in a new cell. Which runs pay
   for the spaces between words, and which get them for free?
3. The spelling charge. It treats every character as equally likely, and real spelling is far more predictable
   than that. If a better spelling model replaced it, whose numbers would move, and which way?

## Further reflection (ungraded)

Where do a BPE model's bits go? Encode a common word and a character name from the test story and compare how many
tokens each becomes.

## Report values

Run this cell and enter D1 and D2 in checkpoint 1. They come from the stories themselves, so only a notebook that has
loaded and split the data produces them.

In [ ]:
D1 = len(set(TRAIN_WORDS))
D2 = len(encode(TEST, MERGES[:1000]))
print("D1 (different words in the training stories) =", D1)
print("D2 (test-story tokens under BPE with 1,000 merges) =", D2)